# Hong Kong Free Press news scraper

This notebook downloads English news articles from [Hong Kong Free Press](https://hongkongfp.com/) for an inclusive date range.

The scraper writes two local outputs:

- Article text files with content only: `C:\Program Files\Studying\coding\RAG_project\data\hk_free_press_news`
- One metadata CSV file for the run: `C:\Program Files\Studying\coding\RAG_project\data\hk_free_press_news_metadata`

The main function is `scrape_hkfp_news(start_date, end_date)`. Use dates in `DD-MM-YYYY` format, for example `scrape_hkfp_news("01-05-2025", "01-05-2025")` for one day.

In [8]:
from __future__ import annotations

import csv
import html
import re
from datetime import date, datetime, time as datetime_time
from html.parser import HTMLParser
from pathlib import Path
from typing import Any
from zoneinfo import ZoneInfo

import requests

PROJECT_ROOT = Path(r"C:\Program Files\Studying\coding\RAG_project")
NEWS_DIR = PROJECT_ROOT / "data" / "hk_free_press_news"
METADATA_DIR = PROJECT_ROOT / "data" / "hk_free_press_news_metadata"

HKFP_API_URL = "https://hongkongfp.com/wp-json/wp/v2/posts"
HK_TIMEZONE = ZoneInfo("Asia/Hong_Kong")


class ArticleTextExtractor(HTMLParser):
    """Convert article HTML into readable plain text with no third-party parser."""

    def __init__(self) -> None:
        super().__init__(convert_charrefs=True)
        self._parts: list[str] = []
        self._skip_depth = 0
        self._block_tags = {"p", "br", "div", "section", "article", "h1", "h2", "h3", "h4", "li", "blockquote"}
        self._skip_tags = {"script", "style", "noscript", "iframe", "svg", "form", "aside"}

    def handle_starttag(self, tag: str, attrs: list[tuple[str, str | None]]) -> None:
        del attrs
        if tag in self._skip_tags:
            self._skip_depth += 1
        elif self._skip_depth == 0 and tag in self._block_tags:
            self._parts.append("\n")

    def handle_endtag(self, tag: str) -> None:
        if tag in self._skip_tags and self._skip_depth:
            self._skip_depth -= 1
        elif self._skip_depth == 0 and tag in self._block_tags:
            self._parts.append("\n")

    def handle_data(self, data: str) -> None:
        if self._skip_depth == 0:
            self._parts.append(data)

    def text(self) -> str:
        raw_text = html.unescape("".join(self._parts))
        lines = [re.sub(r"\s+", " ", line).strip() for line in raw_text.splitlines()]
        return "\n\n".join(line for line in lines if line)


def clean_html_text(rendered_html: str) -> str:
    """Return clean article text from WordPress-rendered HTML."""
    parser = ArticleTextExtractor()
    parser.feed(rendered_html or "")
    parser.close()
    return parser.text()


def parse_scrape_date(value: date | str) -> date:
    """Parse a user date. Preferred input format is DD-MM-YYYY."""
    if isinstance(value, date):
        return value

    for date_format in ("%d-%m-%Y", "%Y-%m-%d"):
        try:
            return datetime.strptime(value, date_format).date()
        except ValueError:
            pass

    raise ValueError("Use date format DD-MM-YYYY, for example '01-05-2025'.")


def parse_wordpress_date(value: str) -> datetime:
    """Parse a WordPress timestamp and convert it to Hong Kong time."""
    parsed = datetime.fromisoformat(value.replace("Z", "+00:00"))
    if parsed.tzinfo is None:
        parsed = parsed.replace(tzinfo=HK_TIMEZONE)
    return parsed.astimezone(HK_TIMEZONE)


def safe_article_filename(value: str, max_length: int = 120) -> str:
    """Make an article title safe for a Windows filename while keeping it readable."""
    value = html.unescape(value).strip()
    value = re.sub(r'[<>:"/\\|?*]', "", value)
    value = re.sub(r"\s+", " ", value).strip(" .")
    return (value[:max_length].strip(" .") or "Untitled article")


def get_embedded_terms(post: dict[str, Any], taxonomy: str) -> list[str]:
    """Read embedded category or tag names from the WordPress API response."""
    terms: list[str] = []
    for group in post.get("_embedded", {}).get("wp:term", []):
        for item in group:
            if item.get("taxonomy") == taxonomy and item.get("name"):
                terms.append(html.unescape(item["name"]))
    return terms


def request_hkfp_posts_between_dates(
    start_date: date,
    end_date: date,
    session: requests.Session,
    per_page: int = 100,
    max_pages: int = 100,
    timeout_seconds: int = 30,
) -> list[dict[str, Any]]:
    """Download candidate posts from the HKFP WordPress API for an inclusive date range."""
    start_hk = datetime.combine(start_date, datetime_time.min, tzinfo=HK_TIMEZONE)
    end_hk = datetime.combine(end_date, datetime_time.max, tzinfo=HK_TIMEZONE)

    posts: list[dict[str, Any]] = []
    for page in range(1, max_pages + 1):
        params = {
            "after": start_hk.isoformat(),
            "before": end_hk.isoformat(),
            "per_page": per_page,
            "page": page,
            "orderby": "date",
            "order": "desc",
            "_embed": "1",
        }
        response = session.get(HKFP_API_URL, params=params, timeout=timeout_seconds)

        # A 400 response after page 1 usually means the requested page is beyond the result count.
        if response.status_code == 400 and page > 1:
            break
        response.raise_for_status()

        page_posts = response.json()
        if not page_posts:
            break

        posts.extend(page_posts)
        total_pages = int(response.headers.get("X-WP-TotalPages", page))
        if page >= total_pages:
            break

    return posts


def scrape_hkfp_news(start_date: date | str, end_date: date | str) -> tuple[list[Path], Path]:
    """
    Scrape Hong Kong Free Press news articles for an inclusive date range.

    Parameters
    ----------
    start_date:
        First date to scrape. Use DD-MM-YYYY, for example '01-01-2025'.
    end_date:
        Last date to scrape. Use DD-MM-YYYY, for example '01-05-2025'.
        For one day, pass the same value for start_date and end_date.

    Returns
    -------
    tuple[list[Path], Path]
        First output: article text file paths saved in NEWS_DIR.
        Second output: one metadata CSV file path saved in METADATA_DIR.
    """
    start = parse_scrape_date(start_date)
    end = parse_scrape_date(end_date)
    if start > end:
        raise ValueError("start_date must be earlier than or equal to end_date.")

    NEWS_DIR.mkdir(parents=True, exist_ok=True)
    METADATA_DIR.mkdir(parents=True, exist_ok=True)

    session = requests.Session()
    session.headers.update(
        {
            # Identify the notebook politely; some websites reject blank/default clients.
            "User-Agent": "RAG-project-news-scraper/1.0 (daily research use)",
            "Accept": "application/json",
        }
    )

    posts = request_hkfp_posts_between_dates(start_date=start, end_date=end, session=session)
    saved_news_files: list[Path] = []
    metadata_rows: list[dict[str, str]] = []
    filename_counts: dict[str, int] = {}

    for post in posts:
        published_hk = parse_wordpress_date(post["date"])
        if not (start <= published_hk.date() <= end):
            continue

        article_title = html.unescape(post.get("title", {}).get("rendered", "Untitled article"))
        article_text = clean_html_text(post.get("content", {}).get("rendered", ""))
        categories = get_embedded_terms(post, "category")
        article_url = post.get("link", "")

        date_prefix = published_hk.strftime("%d-%m-%Y")
        filename_stem = f"{date_prefix}_{safe_article_filename(article_title)}"

        # Keep the requested filename style, but avoid collisions when two articles share a title.
        filename_counts[filename_stem] = filename_counts.get(filename_stem, 0) + 1
        if filename_counts[filename_stem] > 1:
            filename_stem = f"{filename_stem}_{filename_counts[filename_stem]}"

        news_file = NEWS_DIR / f"{filename_stem}.txt"
        news_file.write_text(article_text.strip() + "\n", encoding="utf-8")
        saved_news_files.append(news_file)

        metadata_rows.append(
            {
                "articles": article_title,
                "released_date": date_prefix,
                "categories": "; ".join(categories),
                "url": article_url,
            }
        )

    metadata_csv = METADATA_DIR / f"hkfp_metadata_{start.strftime('%d-%m-%Y')}_to_{end.strftime('%d-%m-%Y')}.csv"
    with metadata_csv.open("w", newline="", encoding="utf-8-sig") as csv_file:
        fieldnames = ["articles", "released_date", "categories", "url"]
        writer = csv.DictWriter(csv_file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(metadata_rows)

    print(f"Saved {len(saved_news_files)} article text files to: {NEWS_DIR}")
    print(f"Saved metadata CSV to: {metadata_csv}")
    return saved_news_files, metadata_csv

## Run scraper

Use `DD-MM-YYYY` dates. The range is inclusive.

For one day, set `start_date` and `end_date` to the same value.

In [10]:
# Example 1: scrape one day.
news_files, metadata_csv = scrape_hkfp_news("01-01-2026", "30-01-2026")

# Example 2: scrape a date range.
# news_files, metadata_csv = scrape_hkfp_news("01-01-2025", "01-05-2025")

print(f"News text files: {len(news_files)}")
print(f"Metadata CSV: {metadata_csv}")

Saved 186 article text files to: C:\Program Files\Studying\coding\RAG_project\data\hk_free_press_news
Saved metadata CSV to: C:\Program Files\Studying\coding\RAG_project\data\hk_free_press_news_metadata\hkfp_metadata_01-01-2026_to_30-01-2026.csv
News text files: 186
Metadata CSV: C:\Program Files\Studying\coding\RAG_project\data\hk_free_press_news_metadata\hkfp_metadata_01-01-2026_to_30-01-2026.csv
